# IMDB Sentiment (Azure ML): Train, Track (MLflow), Register, Deploy, Invoke

This notebook trains a simple sentiment classifier (**TF-IDF + Logistic Regression**) using the IMDB dataset registered as an Azure ML Data Asset: `azureml:imdb_dataset:1`.

It then:
- logs training to **MLflow**
- registers the model in the Azure ML workspace
- deploys the model to a **Managed Online Endpoint** (MLflow no-code deployment)
- invokes the endpoint and prints predictions


In [ ]:
# If needed, uncomment installs (Azure ML docs commonly recommend these for interactive work).
# %pip install -U azure-ai-ml azure-identity mltable mlflow azureml-mlflow pandas scikit-learn


In [ ]:
import datetime
import json

import pandas as pd
import mltable
import mlflow
import mlflow.sklearn

from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, Model
from azure.identity import DefaultAzureCredential

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


## Connect To Workspace

This uses `config.json` in the current directory (Azure ML Studio notebooks commonly provide it).


In [ ]:
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

ws = ml_client.workspaces.get(name=ml_client.workspace_name)
mlflow.set_tracking_uri(ws.mlflow_tracking_uri)
mlflow.set_experiment("imdb-sentiment")

print("Workspace:", ml_client.workspace_name)
print("MLflow tracking URI:", mlflow.get_tracking_uri())


## Load Data From Azure ML Data Asset

Data asset: `azureml:imdb_dataset:1`


In [ ]:
data_asset = ml_client.data.get(name="imdb_dataset", version="1")

# For a CSV file data asset, materialize with mltable.
tbl = mltable.from_delimited_files(paths=[{"file": data_asset.path}])
df = tbl.to_pandas_dataframe()

df = df[["review", "sentiment"]].dropna()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()

df.head()


## Train + Log With MLflow

This logs:
- model hyperparameters
- test accuracy
- the trained model as an **MLflow model** (so it can be deployed with no-code MLflow deployment)


In [ ]:
X = df["review"].astype(str)
y = df["sentiment"].map({"negative": 0, "positive": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf_max_features = 20000
tfidf_ngram_range = (1, 2)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_features=tfidf_max_features, ngram_range=tfidf_ngram_range)),
    ("clf", LogisticRegression(max_iter=1000)),
])

with mlflow.start_run(run_name="tfidf_logreg") as run:
    mlflow.log_param("tfidf_max_features", tfidf_max_features)
    mlflow.log_param("tfidf_ngram_range", str(tfidf_ngram_range))
    mlflow.log_param("model", "logistic_regression")

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    acc = accuracy_score(y_test, preds)
    mlflow.log_metric("test_accuracy", float(acc))

    # Log as an MLflow model (PyFunc).
    input_example = pd.DataFrame({"review": ["This movie was great"]})
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
    )

    run_id = run.info.run_id

print("Run ID:", run_id)
print(f"Test accuracy: {acc:.4f}")


## Register Model In Azure ML

Registers the MLflow model logged in the run as an Azure ML model asset.


In [ ]:
model_name = "imdb-sentiment-mlflow"
model_path_in_run = "model"  # same as artifact_path in mlflow.sklearn.log_model

registered_model = ml_client.models.create_or_update(
    Model(
        name=model_name,
        type=AssetTypes.MLFLOW_MODEL,
        path=f"azureml://jobs/{run_id}/outputs/artifacts/{model_path_in_run}",
        description="IMDB sentiment TF-IDF + LogisticRegression (MLflow model)",
    )
)

print("Registered model:", registered_model.name, registered_model.version)


## Deploy To Managed Online Endpoint (No-Code MLflow Deployment)

Creates:
- a Managed Online Endpoint (auth: key)
- a deployment named `blue`
- routes 100% traffic to `blue`


In [ ]:
endpoint_name = "imdb-sentiment-" + datetime.datetime.now().strftime("%m%d%H%M%S")

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="IMDB sentiment endpoint (MLflow no-code deployment)",
    auth_mode="key",
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Created endpoint:", endpoint_name)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=registered_model,
    instance_type="Standard_D2ds_v5",
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(deployment).result()

# Route all traffic to this deployment
endpoint = ml_client.online_endpoints.get(endpoint_name)
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

endpoint = ml_client.online_endpoints.get(endpoint_name)
print("Scoring URI:", endpoint.scoring_uri)


## Invoke Endpoint (Prediction)

Azure ML online endpoints expect JSON with top-level key `input_data`.

We send a single column `review` with one or more rows.


In [ ]:
sample_request = {
    "input_data": {
        "columns": ["review"],
        "data": [
            ["This movie was fantastic. Great acting and a compelling story."],
            ["Terrible film. I wasted two hours and the plot was nonsense."],
        ],
    }
}

request_file = "sample-request-imdb.json"
with open(request_file, "w") as f:
    json.dump(sample_request, f)

response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    request_file=request_file,
)

print(response)


## (Optional) Call Endpoint With REST

This shows how to call the endpoint URL directly using the endpoint key.


In [ ]:
import requests

endpoint = ml_client.online_endpoints.get(endpoint_name)
keys = ml_client.online_endpoints.get_keys(endpoint_name)

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {keys.primary_key}",
}

resp = requests.post(endpoint.scoring_uri, headers=headers, json=sample_request)
print(resp.status_code)
print(resp.text)
